In [1]:
import pandas as pd
import numpy as np

from ortools.sat.python import cp_model

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
import pandas as pd
import numpy as np

from ortools.sat.python import cp_model

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
decision_data = pd.read_csv(
    "../data/predictions/multi_objective_decision_scores.csv"
)

print("Decision data rows:", len(decision_data))

Decision data rows: 6


In [4]:
print(monthly_plan.columns.tolist())

NameError: name 'monthly_plan' is not defined

In [5]:
decision_data = pd.read_csv(
    "../data/predictions/multi_objective_decision_scores.csv"
)

In [6]:
monthly_plan = pd.read_csv(
    "../data/predictions/monthly_block_plan.csv"
)

print("Monthly plan rows:", len(monthly_plan))
monthly_plan.head()


Monthly plan rows: 4


,plan_sequence,task_id,section_id,department,urgency_tier,planning_period,start_slot,end_slot,start_time,end_time,estimated_duration,required_manpower,maintenance_decision_score,predicted_delay_minutes
0,1,TMS001,NDL-MTJ-01,ENGINEERING,MEDIUM,5,1393,1396,Day 30 00:30,Day 30 02:00,90,8,0.322000,0.0
1,2,TDMS001,GWL-JHS-01,TRACTION,MEDIUM,5,1399,1401,Day 30 03:30,Day 30 04:30,60,5,0.270667,0.0
2,3,SMMS001,MTJ-AGC-01,S&T,MEDIUM,5,1399,1401,Day 30 03:30,Day 30 04:30,45,3,0.274500,0.0
3,4,TMS002,NDL-MTJ-02,ENGINEERING,LOW,5,1401,1403,Day 30 04:30,Day 30 05:30,60,5,0.224667,0.0


In [7]:
print(monthly_plan.columns.tolist())

['plan_sequence', 'task_id', 'section_id', 'department', 'urgency_tier', 'planning_period', 'start_slot', 'end_slot', 'start_time', 'end_time', 'estimated_duration', 'required_manpower', 'maintenance_decision_score', 'predicted_delay_minutes']


In [8]:
monthly_plan[
    [
        "task_id",
        "section_id",
        "department",
        "start_time",
        "end_time",
        "maintenance_decision_score"
    ]
].head(20)

,task_id,section_id,department,start_time,end_time,maintenance_decision_score
0,TMS001,NDL-MTJ-01,ENGINEERING,Day 30 00:30,Day 30 02:00,0.322000
1,TDMS001,GWL-JHS-01,TRACTION,Day 30 03:30,Day 30 04:30,0.270667
2,SMMS001,MTJ-AGC-01,S&T,Day 30 03:30,Day 30 04:30,0.274500
3,TMS002,NDL-MTJ-02,ENGINEERING,Day 30 04:30,Day 30 05:30,0.224667


In [9]:
monthly_plan[
    [
        "task_id",
        "section_id",
        "department",
        "start_time",
        "end_time",
        "maintenance_decision_score"
    ]
].head(20)

,task_id,section_id,department,start_time,end_time,maintenance_decision_score
0,TMS001,NDL-MTJ-01,ENGINEERING,Day 30 00:30,Day 30 02:00,0.322000
1,TDMS001,GWL-JHS-01,TRACTION,Day 30 03:30,Day 30 04:30,0.270667
2,SMMS001,MTJ-AGC-01,S&T,Day 30 03:30,Day 30 04:30,0.274500
3,TMS002,NDL-MTJ-02,ENGINEERING,Day 30 04:30,Day 30 05:30,0.224667


In [10]:
CURRENT_DAY = 8

SLOT_MINUTES = 30
SLOTS_PER_DAY = 48

CURRENT_SLOT = (
    CURRENT_DAY - 1
) * SLOTS_PER_DAY

print("Current planning day:", CURRENT_DAY)
print("Current slot:", CURRENT_SLOT)

Current planning day: 8
Current slot: 336


In [11]:
completed_tasks = monthly_plan[
    monthly_plan["end_slot"] <= CURRENT_SLOT
].copy()

future_tasks = monthly_plan[
    monthly_plan["end_slot"] > CURRENT_SLOT
].copy()

print(
    "Completed/past tasks:",
    len(completed_tasks)
)

print(
    "Future tasks:",
    len(future_tasks)
)

Completed/past tasks: 0
Future tasks: 4


In [12]:
completed_tasks[
    [
        "task_id",
        "section_id",
        "department",
        "start_time",
        "end_time"
    ]
].head(20)

,task_id,section_id,department,start_time,end_time


In [13]:
new_event = {
    "event_id": "EVT_001",
    "event_type": "SIGNAL_FAILURE",
    "section_id": "JHS-BINA-01",
    "department": "S&T",
    "severity": 10,
    "criticality": 10,
    "additional_delay_minutes": 60
}

print(new_event)

{'event_id': 'EVT_001', 'event_type': 'SIGNAL_FAILURE', 'section_id': 'JHS-BINA-01', 'department': 'S&T', 'severity': 10, 'criticality': 10, 'additional_delay_minutes': 60}


In [14]:
affected_tasks = future_tasks[
    future_tasks["section_id"]
    == new_event["section_id"]
].copy()

print(
    "Affected future tasks:",
    len(affected_tasks)
)

display(
    affected_tasks[
        [
            "task_id",
            "section_id",
            "department",
            "start_time",
            "end_time"
        ]
    ]
)

Affected future tasks: 0


,task_id,section_id,department,start_time,end_time


In [15]:
new_task = pd.DataFrame([{
    "task_id": "NEW_EVT_001",
    "section_id": new_event["section_id"],
    "department": new_event["department"],
    "estimated_duration": 60,
    "required_manpower": 4,
    "maintenance_decision_score": 1.0,
    "predicted_delay_minutes": new_event[
        "additional_delay_minutes"
    ],
    "urgency_tier": "CRITICAL"
}])

display(new_task)

,task_id,section_id,department,estimated_duration,required_manpower,maintenance_decision_score,predicted_delay_minutes,urgency_tier
0,NEW_EVT_001,JHS-BINA-01,S&T,60,4,1.0,60,CRITICAL


In [16]:
rolling_candidates = future_tasks[
    [
        "task_id",
        "section_id",
        "department",
        "estimated_duration",
        "required_manpower",
        "maintenance_decision_score",
        "predicted_delay_minutes",
        "urgency_tier"
    ]
].copy()

rolling_candidates = pd.concat(
    [
        rolling_candidates,
        new_task
    ],
    ignore_index=True
)

rolling_candidates = (
    rolling_candidates
    .drop_duplicates(
        subset=["task_id"]
    )
    .reset_index(drop=True)
)

print(
    "Rolling-horizon candidates:",
    len(rolling_candidates)
)

Rolling-horizon candidates: 5


In [17]:
rolling_candidates["duration_slots"] = np.ceil(
    rolling_candidates["estimated_duration"]
    / SLOT_MINUTES
).astype(int)

rolling_candidates["duration_slots"] = (
    rolling_candidates["duration_slots"]
    .clip(lower=1)
)

rolling_candidates[
    [
        "task_id",
        "estimated_duration",
        "duration_slots"
    ]
].head(10)

,task_id,estimated_duration,duration_slots
0,TMS001,90,3
1,TDMS001,60,2
2,SMMS001,45,2
3,TMS002,60,2
4,NEW_EVT_001,60,2


In [18]:
TOTAL_MONTHLY_SLOTS = (
    30 * SLOTS_PER_DAY
)

remaining_slots = (
    TOTAL_MONTHLY_SLOTS
    - CURRENT_SLOT
)

print(
    "Total monthly slots:",
    TOTAL_MONTHLY_SLOTS
)

print(
    "Remaining slots:",
    remaining_slots
)

Total monthly slots: 1440
Remaining slots: 1104


In [19]:
future_block_windows = []

for day in range(
    CURRENT_DAY - 1,
    30
):

    day_start = (
        day * SLOTS_PER_DAY
    )

    future_block_windows.append({
        "day": day + 1,
        "block_id": f"D{day+1}_B1",
        "start_slot": day_start + 1,
        "end_slot": day_start + 6
    })

    future_block_windows.append({
        "day": day + 1,
        "block_id": f"D{day+1}_B2",
        "start_slot": day_start + 7,
        "end_slot": day_start + 11
    })

print(
    "Future maintenance windows:",
    len(future_block_windows)
)

Future maintenance windows: 46


In [20]:
model = cp_model.CpModel()

print(
    "Rolling-horizon CP-SAT model created."
)

Rolling-horizon CP-SAT model created.


In [21]:
task_selected = {}
task_start = {}
task_end = {}

for i, row in rolling_candidates.iterrows():

    duration = int(
        row["duration_slots"]
    )

    task_selected[i] = model.NewBoolVar(
        f"rolling_selected_{i}"
    )

    task_start[i] = model.NewIntVar(
        CURRENT_SLOT,
        TOTAL_MONTHLY_SLOTS - duration,
        f"rolling_start_{i}"
    )

    task_end[i] = model.NewIntVar(
        CURRENT_SLOT,
        TOTAL_MONTHLY_SLOTS,
        f"rolling_end_{i}"
    )

    model.Add(
        task_end[i]
        ==
        task_start[i] + duration
    )

print(
    "Rolling task variables:",
    len(task_selected)
)

Rolling task variables: 5


In [22]:
for i, row in rolling_candidates.iterrows():

    duration = int(
        row["duration_slots"]
    )

    block_choices = []

    for j, block in enumerate(
        future_block_windows
    ):

        start_min = max(
            block["start_slot"],
            CURRENT_SLOT
        )

        start_max = (
            block["end_slot"]
            - duration
        )

        if start_max >= start_min:

            choice = model.NewBoolVar(
                f"rolling_{i}_block_{j}"
            )

            block_choices.append(choice)

            model.Add(
                task_start[i]
                >= start_min
            ).OnlyEnforceIf(choice)

            model.Add(
                task_start[i]
                <= start_max
            ).OnlyEnforceIf(choice)

    model.Add(
        sum(block_choices)
        ==
        task_selected[i]
    )

print(
    "Future block constraints added."
)

Future block constraints added.


In [23]:
for i, row in rolling_candidates.iterrows():

    duration = int(
        row["duration_slots"]
    )

    block_choices = []

    for j, block in enumerate(
        future_block_windows
    ):

        start_min = max(
            block["start_slot"],
            CURRENT_SLOT
        )

        start_max = (
            block["end_slot"]
            - duration
        )

        if start_max >= start_min:

            choice = model.NewBoolVar(
                f"rolling_{i}_block_{j}"
            )

            block_choices.append(choice)

            model.Add(
                task_start[i]
                >= start_min
            ).OnlyEnforceIf(choice)

            model.Add(
                task_start[i]
                <= start_max
            ).OnlyEnforceIf(choice)

    model.Add(
        sum(block_choices)
        ==
        task_selected[i]
    )

print(
    "Future block constraints added."
)

Future block constraints added.


In [24]:
section_intervals = {}

for i, row in rolling_candidates.iterrows():

    duration = int(
        row["duration_slots"]
    )

    section = row["section_id"]

    interval = model.NewOptionalIntervalVar(
        task_start[i],
        duration,
        task_end[i],
        task_selected[i],
        f"rolling_section_{i}"
    )

    section_intervals.setdefault(
        section,
        []
    ).append(interval)

In [25]:
for section, intervals in section_intervals.items():

    model.AddNoOverlap(intervals)

print(
    "Rolling section constraints added."
)

Rolling section constraints added.


In [26]:
MAX_MANPOWER = 12

manpower_intervals = []
manpower_demands = []

for i, row in rolling_candidates.iterrows():

    duration = int(
        row["duration_slots"]
    )

    manpower = int(
        row["required_manpower"]
    )

    interval = model.NewOptionalIntervalVar(
        task_start[i],
        duration,
        task_end[i],
        task_selected[i],
        f"rolling_manpower_{i}"
    )

    manpower_intervals.append(
        interval
    )

    manpower_demands.append(
        manpower
    )

model.AddCumulative(
    manpower_intervals,
    manpower_demands,
    MAX_MANPOWER
)

print(
    "Rolling manpower constraint added."
)

Rolling manpower constraint added.


In [27]:
objective_terms = []

for i, row in rolling_candidates.iterrows():

    priority = int(
        row["maintenance_decision_score"]
        * 1000
    )

    impact = int(
        row["predicted_delay_minutes"]
        * 10
    )

    coefficient = (
        priority
        - impact
    )

    if row["task_id"] == "NEW_EVT_001":

        coefficient += 5000

    objective_terms.append(
        coefficient
        * task_selected[i]
    )

model.Maximize(
    sum(objective_terms)
)

print(
    "Rolling-horizon objective created."
)

Rolling-horizon objective created.


In [28]:
solver = cp_model.CpSolver()

solver.parameters.max_time_in_seconds = 60
solver.parameters.num_search_workers = 8

status = solver.Solve(model)

print(
    "Solver status:",
    solver.StatusName(status)
)

print(
    "Objective value:",
    solver.ObjectiveValue()
)

Solver status: OPTIMAL
Objective value: 6490.0


In [29]:
rolling_plan_rows = []

for i, row in rolling_candidates.iterrows():

    if solver.Value(
        task_selected[i]
    ) == 1:

        result = row.copy()

        result["start_slot"] = (
            solver.Value(
                task_start[i]
            )
        )

        result["end_slot"] = (
            solver.Value(
                task_end[i]
            )
        )

        rolling_plan_rows.append(
            result
        )

rolling_plan = pd.DataFrame(
    rolling_plan_rows
)

print(
    "Tasks scheduled after re-optimization:",
    len(rolling_plan)
)

Tasks scheduled after re-optimization: 5


In [30]:
def slot_to_datetime(slot):

    day = (
        slot // SLOTS_PER_DAY
    )

    slot_in_day = (
        slot % SLOTS_PER_DAY
    )

    total_minutes = (
        slot_in_day
        * SLOT_MINUTES
    )

    hours = total_minutes // 60
    minutes = total_minutes % 60

    return (
        f"Day {day + 1} "
        f"{hours:02d}:{minutes:02d}"
    )

In [31]:
rolling_plan["start_time"] = (
    rolling_plan["start_slot"]
    .apply(slot_to_datetime)
)

rolling_plan["end_time"] = (
    rolling_plan["end_slot"]
    .apply(slot_to_datetime)
)


In [32]:
emergency_result = rolling_plan[
    rolling_plan["task_id"]
    == "NEW_EVT_001"
]

display(
    emergency_result[
        [
            "task_id",
            "section_id",
            "department",
            "start_time",
            "end_time",
            "urgency_tier"
        ]
    ]
)

,task_id,section_id,department,start_time,end_time,urgency_tier
4,NEW_EVT_001,JHS-BINA-01,S&T,Day 9 00:30,Day 9 01:30,CRITICAL


In [33]:
original_task_ids = set(
    future_tasks["task_id"]
)

new_task_ids = set(
    rolling_plan["task_id"]
)

added_tasks = (
    new_task_ids
    - original_task_ids
)

removed_tasks = (
    original_task_ids
    - new_task_ids
)

print(
    "Added tasks:",
    added_tasks
)

print(
    "Tasks no longer scheduled:",
    removed_tasks
)

Added tasks: {'NEW_EVT_001'}
Tasks no longer scheduled: set()


In [34]:
rolling_plan.to_csv(
    "../data/predictions/rolling_horizon_plan.csv",
    index=False
)

print(
    "Saved:",
    "../data/predictions/rolling_horizon_plan.csv"
)

Saved: ../data/predictions/rolling_horizon_plan.csv


In [35]:
print(
    "===== ROLLING-HORIZON SUMMARY ====="
)

print(
    "Current planning day:",
    CURRENT_DAY
)

print(
    "Frozen completed tasks:",
    len(completed_tasks)
)

print(
    "Original future tasks:",
    len(future_tasks)
)

print(
    "New scheduled tasks:",
    len(rolling_plan)
)

print(
    "Emergency task scheduled:",
    "NEW_EVT_001"
    in set(rolling_plan["task_id"])
)

===== ROLLING-HORIZON SUMMARY =====
Current planning day: 8
Frozen completed tasks: 0
Original future tasks: 4
New scheduled tasks: 5
Emergency task scheduled: True
